# PaperMind Full-Text Fetcher

**Amac:** OpenAlex OA makalelerinin tam metinlerini (PDF) Drive'a indirir.

**3 Colab paralel:** Her notebook bir SHARD calistirir.

| Shard | Email | Hesap |
|-------|-------|-------|
| 0 | ofrencber@gantep.edu.tr | Colab 1 |
| 1 | aaksoy@gantep.edu.tr | Colab 2 |
| 2 | dr.ofrencber@gaziantep.edu.tr | Colab 3 |

**Pipeline:**
1. OpenAlex API -> OA makale kesfet (is_oa=true, 2015+, article)
2. Abstract'ten keyword-based yontem tespiti
3. PDF indir -> Drive
4. Ilerleme durumu fetch_state.json'da

In [ ]:
# ============================================================
# AYAR — Her Colab'da sadece bu hucreyi degistir
# ============================================================
SHARD_ID = 0  # 0, 1, veya 2
TOTAL_SHARDS = 3

EMAILS = {
    0: "ofrencber@gantep.edu.tr",
    1: "aaksoy@gantep.edu.tr",
    2: "dr.ofrencber@gaziantep.edu.tr",
}
MY_EMAIL = EMAILS[SHARD_ID]

# Drive klasoru
DRIVE_ROOT = "/content/drive/MyDrive/papermind_fulltext"

# Indirme hedefi
MIN_YEAR = 2015
MAX_YEAR = 2026

print(f"Shard {SHARD_ID}/{TOTAL_SHARDS} — email: {MY_EMAIL}")

In [ ]:
# ============================================================
# Drive mount + dizin olustur
# ============================================================
from google.colab import drive
drive.mount("/content/drive")

import os
for d in ["manifest", f"pdfs/shard_{SHARD_ID}", "parsed", "errors", "state"]:
    os.makedirs(f"{DRIVE_ROOT}/{d}", exist_ok=True)
print("Dizinler hazir.")

In [ ]:
# ============================================================
# Bagimliliklar
# ============================================================
!pip install -q requests tqdm pandas

In [ ]:
import requests
import pandas as pd
import json
import time
import re
from pathlib import Path
from tqdm.auto import tqdm
from urllib.parse import urlparse
from collections import defaultdict
from datetime import datetime


# ============================================================
# Logger — hem konsola hem Drive'a yazar
# ============================================================
class DriveLogger:
    """Konsol + Drive dosyasina ayni anda yazar."""
    def __init__(self, log_path: str):
        self.log_path = log_path
        os.makedirs(os.path.dirname(log_path), exist_ok=True)

    def log(self, msg: str, level: str = "INFO"):
        ts = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        line = f"[{ts}] [{level}] {msg}"
        print(line)
        with open(self.log_path, "a") as f:
            f.write(line + "\n")

    def info(self, msg): self.log(msg, "INFO")
    def warn(self, msg): self.log(msg, "WARN")
    def error(self, msg): self.log(msg, "ERROR")
    def checkpoint(self, msg): self.log(msg, "CHECKPOINT")


def _safe_read_csv(path: str) -> pd.DataFrame | None:
    """CSV oku — dosya yoksa veya boşsa None dondurur."""
    if not os.path.exists(path) or os.path.getsize(path) < 10:
        return None
    try:
        df = pd.read_csv(path)
        return df if not df.empty else None
    except Exception:
        return None


logger = DriveLogger(f"{DRIVE_ROOT}/state/shard_{SHARD_ID}.log")
logger.info(f"Oturum basladi — Shard {SHARD_ID}, email: {MY_EMAIL}")

## 1. OpenAlex SSCI Alanları

OpenAlex domain/field hiyerarşisi üzerinden SSCI-karşılığı alanlar.

In [ ]:
# ============================================================
# OpenAlex SSCI-karsiligi alanlar (dogrulanmis ID'ler)
# ============================================================
# Filter'da SAYI kullanilir: primary_topic.field.id:33
# TAM URL KULLANILMAZ (0 sonuc doner)
#
# Kaynak: https://api.openalex.org/fields?per_page=50
# ============================================================

UNIQUE_FIELDS = {
    "social_sciences":      33,  # 53M works
    "arts_and_humanities":  12,  # 27M works
    "economics":            20,  # 13M works
    "business":             14,  # 9.3M works
    "psychology":           32,  # 9M works
    "health_professions":   36,  # 7.9M works
    "environmental_sci":    23,  # 17M works
    "decision_sciences":    18,  # 3.5M works
    "nursing":              29,  # 1.3M works
}

print(f"{len(UNIQUE_FIELDS)} alan:")
for k, v in UNIQUE_FIELDS.items():
    print(f"  {k}: field_id={v}")

## 2. Keyword-Based Yontem Tespiti

In [ ]:
# ============================================================
# Abstract'ten yontem siniflandirma (keyword-based)
# ============================================================

METHOD_PATTERNS = {
    "meta_analysis": [
        r"\bmeta[- ]?analy",
        r"\bsystematic review\b",
        r"\bprisma\b",
        r"\beffect size[s]?\b",
        r"\bheterogeneity\b",
        r"\bfunnel plot\b",
        r"\bfixed.effect\b",
        r"\brandom.effect[s]? model\b",
    ],
    "qualitative": [
        r"\bqualitative\b",
        r"\bethnograph",
        r"\bgrounded theory\b",
        r"\bphenomenolog",
        r"\bthematic analysis\b",
        r"\bcontent analysis\b",
        r"\bnarrative\s+(analysis|inquiry)\b",
        r"\bfocus groups?\b",
        r"\bin-depth interviews?\b",
        r"\bsemi[- ]?structured interviews?\b",
        r"\bcase stud(y|ies)\b",
        r"\bdiscourse analysis\b",
        r"\binterviews?\s+(were|was)\s+conducted\b",
        r"\bparticipant observ",
        r"\bopen[- ]?ended\s+(question|interview)",
    ],
    "mixed_methods": [
        r"\bmixed[- ]?method",
        r"\bmulti[- ]?method\b",
        r"\btriangulat",
        r"\bsequential explanatory\b",
        r"\bconvergent design\b",
    ],
    "review": [
        r"\bliterature review\b",
        r"\bscoping review\b",
        r"\bnarrative review\b",
        r"\bcritical review\b",
        r"\bumbrella review\b",
        r"\bbibliometric\b",
        r"\bscientometric\b",
    ],
    "quantitative": [
        r"\bregression\b",
        r"\bstructural equation\b",
        r"\bSEM\b",
        r"\banova\b",
        r"\bmanova\b",
        r"\bt[- ]?test\b",
        r"\bchi[- ]?square\b",
        r"\bcorrelation\b",
        r"\bfactor analysis\b",
        r"\bcluster analysis\b",
        r"\bpanel data\b",
        r"\btime series\b",
        r"\blogistic regression\b",
        r"\bmulti[- ]?level\b",
        r"\bhierarchical linear\b",
        r"\bDEA\b",
        r"\bAHP\b",
        r"\bTOPSIS\b",
        r"\bMCDM\b",
        r"\bsurvey\b",
        r"\bquestionnaire\b",
        r"\bexperiment\b",
        r"\brandom\s*i[sz]ed\s*(control)?\s*trial\b",
        r"\bRCT\b",
        r"\bmachine learning\b",
        r"\bdeep learning\b",
        r"\bneural network\b",
        r"\brandom forest\b",
        r"\bGMM\b",
        r"\bdifference.in.difference\b",
        r"\binstrumental variable\b",
        r"\bpropensity score\b",
    ],
}

# Oncelik: meta_analysis > mixed > qualitative > review > quantitative
METHOD_PRIORITY = ["meta_analysis", "mixed_methods", "qualitative", "review", "quantitative"]


def classify_method(abstract: str) -> str:
    """Abstract'ten yontem siniflandirmasi. Eger hicbiri eslesmazse 'unclassified'."""
    if not abstract:
        return "unclassified"
    text = abstract.lower()
    hits = set()
    for method, patterns in METHOD_PATTERNS.items():
        for pat in patterns:
            if re.search(pat, text, re.IGNORECASE):
                hits.add(method)
                break
    if not hits:
        return "unclassified"
    # Oncelik sirasina gore sec
    for m in METHOD_PRIORITY:
        if m in hits:
            return m
    return "unclassified"


# Test
assert classify_method("We conducted a meta-analysis of 50 studies") == "meta_analysis"
assert classify_method("A regression model was estimated using panel data") == "quantitative"
assert classify_method("Semi-structured interviews were conducted") == "qualitative"
assert classify_method("Focus groups were used to collect data") == "qualitative"
assert classify_method("A case study approach was adopted") == "qualitative"
assert classify_method("This scoping review examines") == "review"
assert classify_method("") == "unclassified"
print("Yontem siniflandirma testleri gecti.")

## 3. OpenAlex OA Makale Kesfet (Faz 1)

In [ ]:
# ============================================================
# OpenAlex API — cursor-based paging ile OA makale cek
# ============================================================

OPENALEX_API = "https://api.openalex.org/works"
PER_PAGE = 200  # max 200
RATE_LIMIT_SLEEP = 0.1  # polite pool ile cok rahat


def _invert_abstract(inv_index: dict) -> str:
    """OpenAlex inverted abstract index -> duz metin."""
    if not inv_index:
        return ""
    pos_word = []
    for word, positions in inv_index.items():
        for pos in positions:
            pos_word.append((pos, word))
    pos_word.sort()
    return " ".join(w for _, w in pos_word)


def fetch_oa_papers_for_field(
    field_name: str,
    field_id: int,
    email: str,
    min_year: int = 2015,
    max_year: int = 2026,
    max_papers: int = 50_000,
) -> list[dict]:
    """Bir alan icin OA makaleleri cursor paging ile ceker."""
    filter_str = (
        f"primary_topic.field.id:{field_id},"
        f"is_oa:true,"
        f"type:article,"
        f"has_abstract:true,"
        f"publication_year:{min_year}-{max_year},"
        f"cited_by_count:>0"
    )
    params = {
        "filter": filter_str,
        "select": (
            "id,doi,title,publication_year,cited_by_count,"
            "primary_topic,open_access,abstract_inverted_index,"
            "authorships,primary_location"
        ),
        "per_page": PER_PAGE,
        "cursor": "*",
        "mailto": email,
        "sort": "cited_by_count:desc",
    }

    papers = []
    page = 0
    consecutive_errors = 0
    MAX_CONSECUTIVE_ERRORS = 5

    logger.info(f"BASLIYOR: {field_name} (field_id={field_id})")
    logger.info(f"  filter: {filter_str}")

    while True:
        try:
            resp = requests.get(OPENALEX_API, params=params, timeout=30)
            resp.raise_for_status()
            data = resp.json()
            consecutive_errors = 0

            # Ilk sayfada meta bilgisini logla
            if page == 0:
                meta = data.get("meta", {})
                total_count = meta.get("count", 0)
                logger.info(f"  {field_name}: API toplam {total_count:,} sonuc bildirdi")
                if total_count == 0:
                    logger.warn(f"  {field_name}: 0 sonuc — filter veya field_id kontrol edin")
                    logger.warn(f"  Yanit meta: {json.dumps(meta)}")
                    break

        except Exception as e:
            consecutive_errors += 1
            logger.error(f"{field_name} sayfa {page}: {e} (ardisik hata: {consecutive_errors})")
            if consecutive_errors >= MAX_CONSECUTIVE_ERRORS:
                logger.error(f"{field_name}: {MAX_CONSECUTIVE_ERRORS} ardisik hata, alan atlaniyor!")
                break
            time.sleep(5 * consecutive_errors)
            continue

        results = data.get("results", [])
        if not results:
            break

        for r in results:
            oa = r.get("open_access", {}) or {}
            oa_url = oa.get("oa_url", "")
            if not oa_url:
                continue

            abstract = _invert_abstract(r.get("abstract_inverted_index"))
            pt = r.get("primary_topic") or {}
            loc = r.get("primary_location") or {}
            source = loc.get("source") or {}

            papers.append({
                "openalex_id": r.get("id", "").replace("https://openalex.org/", ""),
                "doi": (r.get("doi") or "").replace("https://doi.org/", ""),
                "title": r.get("title", ""),
                "year": r.get("publication_year"),
                "cited_by_count": r.get("cited_by_count", 0),
                "oa_url": oa_url,
                "oa_status": oa.get("oa_status", ""),
                "field": field_name,
                "topic": pt.get("display_name", ""),
                "subfield": (pt.get("subfield") or {}).get("display_name", ""),
                "journal": source.get("display_name", ""),
                "abstract": abstract,
                "method_type": classify_method(abstract),
            })

        page += 1
        if page % 10 == 0:
            logger.info(f"  {field_name}: {len(papers):,} makale ({page} sayfa)")

        if len(papers) >= max_papers:
            logger.info(f"  {field_name}: {max_papers:,} limite ulasti.")
            break

        next_cursor = data.get("meta", {}).get("next_cursor")
        if not next_cursor:
            break
        params["cursor"] = next_cursor
        time.sleep(RATE_LIMIT_SLEEP)

    logger.checkpoint(f"{field_name}: TOPLAM {len(papers):,} OA makale, {page} sayfa")
    return papers


logger.info("fetch_oa_papers_for_field() hazir.")

In [ ]:
# ============================================================
# Tum alanlari cek — shard'a dusenleri isle
# ============================================================

MANIFEST_PATH = f"{DRIVE_ROOT}/manifest/oa_manifest_shard_{SHARD_ID}.csv"
STATE_PATH = f"{DRIVE_ROOT}/state/fetch_state_shard_{SHARD_ID}.json"

# State yukle (kaldigi yerden devam)
if os.path.exists(STATE_PATH):
    with open(STATE_PATH) as f:
        state = json.load(f)
else:
    state = {"completed_fields": [], "total_papers": 0}

# RESET: Eger onceki calisma 0 makale ile "tamamlandi" isaretlediyse sifirla
if state["total_papers"] == 0 and len(state["completed_fields"]) > 0:
    logger.warn(f"Onceki calisma 0 makale ile {len(state['completed_fields'])} alan tamamladi — STATE SIFIRLANDI")
    state = {"completed_fields": [], "total_papers": 0}
    with open(STATE_PATH, "w") as f:
        json.dump(state, f, indent=2)
    # Bos manifest'i de sil
    if os.path.exists(MANIFEST_PATH):
        os.remove(MANIFEST_PATH)

# Alanlari shard'lara bol
field_list = list(UNIQUE_FIELDS.items())
my_fields = [(n, fid) for i, (n, fid) in enumerate(field_list) if i % TOTAL_SHARDS == SHARD_ID]

print(f"Shard {SHARD_ID} alanlari ({len(my_fields)}):")
for n, _ in my_fields:
    status = "TAMAMLANDI" if n in state["completed_fields"] else "BEKLIYOR"
    print(f"  {n}: {status}")

In [ ]:
# ============================================================
# Faz 1: OA makale kesfet + manifest olustur
# ============================================================

all_papers = []

# Onceden indirilen manifest varsa yukle
existing_df = _safe_read_csv(MANIFEST_PATH)
if existing_df is not None:
    all_papers = existing_df.to_dict("records")
    logger.info(f"Mevcut manifest yuklendi: {len(all_papers):,} makale")

for field_name, field_id in my_fields:
    if field_name in state["completed_fields"]:
        logger.info(f"ATLANDI (zaten tamamlandi): {field_name}")
        continue

    papers = fetch_oa_papers_for_field(
        field_name=field_name,
        field_id=field_id,
        email=MY_EMAIL,
        min_year=MIN_YEAR,
        max_year=MAX_YEAR,
        max_papers=50_000,
    )
    all_papers.extend(papers)

    # State guncelle
    state["completed_fields"].append(field_name)
    state["total_papers"] = len(all_papers)
    with open(STATE_PATH, "w") as f:
        json.dump(state, f, indent=2)

    # Manifest'i her alan sonunda kaydet (crash-safe)
    df = pd.DataFrame(all_papers)
    df.to_csv(MANIFEST_PATH, index=False)
    logger.checkpoint(f"Manifest kaydedildi: {len(all_papers):,} toplam ({field_name} tamamlandi)")

logger.checkpoint(f"=== FAZ 1 TAMAMLANDI: {len(all_papers):,} OA makale ===")

In [ ]:
# ============================================================
# Manifest istatistikleri
# ============================================================

df = _safe_read_csv(MANIFEST_PATH)
if df is None:
    print("Manifest henuz olusturulmadi. Once Faz 1'i calistirin.")
else:
    print(f"Toplam: {len(df):,}")
    print(f"\nAlan dagilimi:")
    print(df["field"].value_counts().to_string())
    print(f"\nYontem dagilimi:")
    print(df["method_type"].value_counts().to_string())
    print(f"\nYil dagilimi:")
    print(df["year"].value_counts().sort_index().to_string())
    print(f"\nOA status:")
    print(df["oa_status"].value_counts().to_string())

## 4. PDF Indirme (Faz 2)

In [ ]:
# ============================================================
# Domain-bazli rate limiter
# ============================================================

class DomainThrottler:
    """Her domain icin son istek zamanini takip eder."""
    def __init__(self, min_interval: float = 1.0):
        self.min_interval = min_interval
        self._last_request: dict[str, float] = defaultdict(float)

    def wait(self, url: str):
        domain = urlparse(url).netloc
        elapsed = time.time() - self._last_request[domain]
        if elapsed < self.min_interval:
            time.sleep(self.min_interval - elapsed)
        self._last_request[domain] = time.time()


throttler = DomainThrottler(min_interval=1.0)
print("DomainThrottler hazir (1 req/sn per domain).")

In [ ]:
# ============================================================
# PDF indirme fonksiyonu
# ============================================================

PDF_DIR = f"{DRIVE_ROOT}/pdfs/shard_{SHARD_ID}"
ERROR_LOG = f"{DRIVE_ROOT}/errors/failed_shard_{SHARD_ID}.csv"
DOWNLOAD_STATE = f"{DRIVE_ROOT}/state/download_state_shard_{SHARD_ID}.json"

HEADERS = {
    "User-Agent": f"PaperMind/1.0 (mailto:{MY_EMAIL}; academic research)",
    "Accept": "application/pdf,*/*",
}

CHUNK_SIZE = 64 * 1024  # 64 KB


def download_pdf(openalex_id: str, oa_url: str, pdf_dir: str) -> tuple[bool, str]:
    """PDF indir. (success, error_msg) dondurur."""
    pdf_path = f"{pdf_dir}/{openalex_id}.pdf"

    # Zaten varsa atla
    if os.path.exists(pdf_path) and os.path.getsize(pdf_path) > 1000:
        return True, "already_exists"

    throttler.wait(oa_url)

    tmp_path = f"{pdf_path}.tmp"
    try:
        resp = requests.get(
            oa_url,
            headers=HEADERS,
            timeout=60,
            stream=True,
            allow_redirects=True,
        )

        if resp.status_code == 429:
            time.sleep(30)
            return False, "rate_limited"

        if resp.status_code != 200:
            return False, f"http_{resp.status_code}"

        content_type = resp.headers.get("content-type", "")
        if "text/html" in content_type and "pdf" not in oa_url.lower():
            resp.close()
            return False, "html_not_pdf"

        # Stream ile chunk yazma (bellek tasarrufu)
        total_written = 0
        is_first_chunk = True
        with open(tmp_path, "wb") as f:
            for chunk in resp.iter_content(chunk_size=CHUNK_SIZE):
                if not chunk:
                    continue
                # Ilk chunk'ta PDF magic bytes kontrolu
                if is_first_chunk:
                    is_first_chunk = False
                    if not chunk[:5] == b"%PDF-" and "pdf" not in oa_url.lower():
                        resp.close()
                        os.remove(tmp_path)
                        return False, "not_pdf_content"
                f.write(chunk)
                total_written += len(chunk)

        # Boyut kontrolu
        if total_written < 1000:
            os.remove(tmp_path)
            return False, "too_small"

        os.rename(tmp_path, pdf_path)
        return True, "ok"

    except requests.exceptions.Timeout:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        return False, "timeout"
    except Exception as e:
        if os.path.exists(tmp_path):
            os.remove(tmp_path)
        return False, str(e)[:100]


print("download_pdf() hazir.")

In [ ]:
# ============================================================
# Toplu PDF indirme
# ============================================================

df = _safe_read_csv(MANIFEST_PATH)
if df is None:
    logger.error("Manifest bos veya yok — once Faz 1'i calistirin.")
    raise SystemExit("Manifest bos.")

logger.info(f"Manifest yuklendi: {len(df):,} makale")

# Download state yukle
if os.path.exists(DOWNLOAD_STATE):
    with open(DOWNLOAD_STATE) as f:
        dl_state = json.load(f)
    logger.info(f"Onceki state yuklendi: {len(dl_state['downloaded'])} indirilmis, index={dl_state['last_index']}")
else:
    dl_state = {"downloaded": [], "failed": [], "last_index": 0}

downloaded_set = set(dl_state["downloaded"])
errors = []

start_idx = dl_state["last_index"]
success_count = len(downloaded_set)
fail_count = len(dl_state["failed"])
skip_count = 0

SAVE_EVERY = 100
CHECKPOINT_EVERY = 500

logger.info(f"PDF indirme basliyor — index {start_idx}'ten devam")

for idx in tqdm(range(start_idx, len(df)), desc="PDF indirme"):
    row = df.iloc[idx]
    oa_id = row["openalex_id"]

    if oa_id in downloaded_set:
        skip_count += 1
        continue

    success, msg = download_pdf(oa_id, row["oa_url"], PDF_DIR)

    if success:
        success_count += 1
        downloaded_set.add(oa_id)
        dl_state["downloaded"].append(oa_id)
    else:
        fail_count += 1
        dl_state["failed"].append({"id": oa_id, "url": row["oa_url"], "error": msg})
        errors.append({"openalex_id": oa_id, "oa_url": row["oa_url"], "error": msg})
        if msg == "rate_limited":
            logger.warn(f"Rate limited: {urlparse(row['oa_url']).netloc} — 30sn bekleniyor")

    # Periyodik state kaydi
    if (idx + 1) % SAVE_EVERY == 0:
        dl_state["last_index"] = idx + 1
        with open(DOWNLOAD_STATE, "w") as f:
            json.dump(dl_state, f)
        if errors:
            pd.DataFrame(errors).to_csv(ERROR_LOG, index=False)

    # Checkpoint logu
    if (idx + 1) % CHECKPOINT_EVERY == 0:
        logger.checkpoint(
            f"PDF [{idx+1:,}/{len(df):,}] "
            f"basarili={success_count:,} basarisiz={fail_count:,} atlandi={skip_count:,}"
        )

# Son kayit
dl_state["last_index"] = len(df)
with open(DOWNLOAD_STATE, "w") as f:
    json.dump(dl_state, f)
if errors:
    pd.DataFrame(errors).to_csv(ERROR_LOG, index=False)

logger.checkpoint(
    f"=== FAZ 2 TAMAMLANDI === "
    f"basarili={success_count:,} basarisiz={fail_count:,} atlandi={skip_count:,}"
)

In [ ]:
# ============================================================
# Indirme sonrasi istatistik
# ============================================================

import glob as globmod

pdfs = globmod.glob(f"{PDF_DIR}/*.pdf")
if not pdfs:
    print("Henuz PDF indirilmedi.")
else:
    total_size_gb = sum(os.path.getsize(p) for p in pdfs) / (1024**3)
    print(f"PDF sayisi: {len(pdfs):,}")
    print(f"Toplam boyut: {total_size_gb:.1f} GB")
    print(f"Ortalama boyut: {total_size_gb * 1024 / len(pdfs):.1f} MB")

err_df = _safe_read_csv(ERROR_LOG)
if err_df is not None:
    print(f"\nHata dagilimi ({len(err_df):,} toplam):")
    print(err_df["error"].value_counts().head(10).to_string())

## 5. Sonraki Adimlar

1. 3 shard manifest'i birlestir: `pd.concat([shard_0, shard_1, shard_2])`
2. GROBID ile yapisal parse (ayri notebook)
3. Alan x yontem x kalite dengeleme
4. Hakemlik / juri simulasyonu veri seti olustur